# Qwen Image Edit: Birchbark Clearing

Load `Qwen/Qwen-Image-Edit-2511` once, then reuse the in-memory pipeline for multiple bark-clearing experiments.

Run this notebook with the project venv kernel from `.venv-qwen-image`.

In [ ]:
from pathlib import Path
import os
import random
import sys

# Keep Qwen weights in the shared workspace cache, not under the project tree.
os.environ.setdefault("HF_HOME", "/workspace-SR006.nfs3/.cache/huggingface")
os.environ.setdefault("HF_HUB_CACHE", "/workspace-SR006.nfs3/.cache/huggingface/hub")

import torch
from PIL import Image, ImageDraw, ImageFilter, ImageFont
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUTPUT_DIR = ROOT / "reports/figs/qwen_image_clear_bark_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 4-bit NF4 quantized Qwen-Image-Edit-2511. The full BF16 model OOMs on this setup.
MODEL_ID = "toandev/Qwen-Image-Edit-2511-4bit"
print("root:", ROOT)
print("output:", OUTPUT_DIR)
print("hf cache:", os.environ["HF_HUB_CACHE"])
print("model:", MODEL_ID)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available(), torch.cuda.device_count())

In [ ]:
BARK_CLEARING_PROMPT = """I gave you birchbark containing ancient slavic symbols. The symbols are carved out on the wood, leaving dark contrast lines. Clear those symbols, returning only clear bark (wooden plate). Very important: The color of new clear plate should match average color of the original, so for example if the original plate mostly dark - the new one should also be mostly dark, and for more light ones the same. One more important thing - the new plate should look "oldish", so make it to contain some carves, roughness, etc. Ensure every symbol is gone and you gave me the clear birch. The form and structure of original birchbark must stay intact, DO NOT CHANGE IT. If there is a scale downside the image remove it entirely"""

NEGATIVE_PROMPT = (
    "letters, text, inscription, symbols, glyphs, black ink, carved marks, ruler, "
    "scale, label, caption"
)

# Tweak these between runs without reloading the model.
INPUT_IMAGE = ROOT / "data/raw/gramoty/documents/novgorod__137/images/photo_novgorod_0137_1.jpg_thumb-large.jpeg"
PROMPT = BARK_CLEARING_PROMPT
STEPS = 30
TRUE_CFG_SCALE = 4.0
GUIDANCE_SCALE = 1.0
SEED = 42

# Memory knobs. 4-bit should fit; offload remains enabled as an extra guard.
CPU_OFFLOAD = False
MAX_INPUT_SIDE = 1024

In [ ]:
# Run once per kernel session. Keep this cell's output alive and re-run later cells for new images.
from diffusers import QwenImageEditPlusPipeline

if "pipe" not in globals():
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    pipe = QwenImageEditPlusPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
    )

    if hasattr(pipe, "enable_vae_tiling"):
        pipe.enable_vae_tiling()
    if hasattr(pipe, "enable_vae_slicing"):
        pipe.enable_vae_slicing()

    if torch.cuda.is_available() and CPU_OFFLOAD:
        pipe.enable_model_cpu_offload()
        device = "cuda + model CPU offload"
    else:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        pipe = pipe.to(device)

    pipe.set_progress_bar_config(disable=False)
    print(f"loaded {MODEL_ID} on {device} with {dtype}")
else:
    print("pipeline already loaded; reusing it")

In [ ]:
def _resize_for_qwen(image: Image.Image, max_side: int | None) -> Image.Image:
    if max_side is None or max(image.size) <= max_side:
        return image
    scale = max_side / max(image.size)
    size = (round(image.width * scale), round(image.height * scale))
    return image.resize(size, Image.Resampling.LANCZOS)


def _preview(image: Image.Image, max_width: int = 512) -> Image.Image:
    if image.width <= max_width:
        return image
    return image.resize((max_width, int(image.height * max_width / image.width)))


def clear_bark(
    input_image: str | Path,
    *,
    prompt: str = BARK_CLEARING_PROMPT,
    negative_prompt: str = NEGATIVE_PROMPT,
    steps: int = STEPS,
    true_cfg_scale: float = TRUE_CFG_SCALE,
    guidance_scale: float = GUIDANCE_SCALE,
    seed: int = SEED,
    max_input_side: int | None = MAX_INPUT_SIDE,
    output_dir: Path = OUTPUT_DIR,
) -> Image.Image:
    input_image = Path(input_image)
    image = Image.open(input_image).convert("RGB")
    image = _resize_for_qwen(image, max_input_side)
    print("input size:", image.size)
    display(_preview(image))

    # With model CPU offload, the pipeline controls device placement internally.
    generator_device = "cuda" if torch.cuda.is_available() and not CPU_OFFLOAD else "cpu"
    generator = torch.Generator(device=generator_device).manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    with torch.inference_mode():
        result = pipe(
            image=[image],
            prompt=prompt,
            generator=generator,
            true_cfg_scale=true_cfg_scale,
            negative_prompt=negative_prompt,
            num_inference_steps=steps,
            guidance_scale=guidance_scale,
            num_images_per_prompt=1,
        ).images[0]

    if torch.cuda.is_available():
        peak_gb = torch.cuda.max_memory_allocated() / 1024**3
        print(f"peak allocated VRAM: {peak_gb:.2f} GB")

    out_path = output_dir / f"{input_image.stem}_qwen_edit_2511_clear_seed{seed}.png"
    result.save(out_path)
    print("saved:", out_path)
    display(_preview(result))
    return result

In [ ]:
# Edit INPUT_IMAGE above, then run this cell repeatedly.
# The model stays loaded as long as the notebook kernel stays alive.

clear = clear_bark(
    INPUT_IMAGE,
    prompt=PROMPT,
    steps=STEPS,
    true_cfg_scale=TRUE_CFG_SCALE,
    guidance_scale=GUIDANCE_SCALE,
    seed=SEED,
)

## Optional: Engrave Exact Synthetic Text

This step does not ask the diffusion model to render letters. It composites deterministic text as incised strokes, so OCR labels remain exact.

In [ ]:
def _load_font(font_path: str | Path | None = None, font_size: int = 48):
    if font_path is not None:
        return ImageFont.truetype(str(font_path), font_size)
    for candidate in (
        "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ):
        path = Path(candidate)
        if path.exists():
            return ImageFont.truetype(str(path), font_size)
    return ImageFont.load_default()


def engrave_text(
    image: Image.Image,
    text: str,
    *,
    font_path: str | Path | None = None,
    font_size: int = 48,
    margin: int = 80,
    strength: float = 1.0,
    output_path: str | Path | None = None,
) -> Image.Image:
    bg = image.convert("RGBA")
    font = _load_font(font_path, font_size)
    mask = Image.new("L", bg.size, 0)
    draw = ImageDraw.Draw(mask)

    y = margin
    for line in [line for line in text.splitlines() if line.strip()]:
        draw.text((margin, y), line, font=font, fill=235)
        bbox = draw.textbbox((margin, y), line, font=font)
        y = bbox[3] + max(8, font_size // 3)

    groove = mask.filter(ImageFilter.GaussianBlur(radius=0.45))
    shadow = Image.new("RGBA", bg.size, (20, 15, 10, 0))
    shadow.putalpha(groove.point(lambda v: int(v * 0.45 * strength)))
    highlight = Image.new("RGBA", bg.size, (235, 220, 188, 0))
    highlight.putalpha(groove.point(lambda v: int(v * 0.20 * strength)))
    dark = Image.new("RGBA", bg.size, (18, 13, 9, 0))
    dark.putalpha(groove.point(lambda v: int(v * 0.14 * strength)))

    bg.alpha_composite(highlight, (-2, -2))
    bg.alpha_composite(shadow, (2, 2))
    bg.alpha_composite(dark)
    result = bg.convert("RGB")
    if output_path is not None:
        result.save(output_path)
        print("saved:", output_path)
    display(result.resize((min(512, result.width), int(result.height * min(512, result.width) / result.width))))
    return result

In [ ]:
# Example after you have a `clear` image from the previous section:
# engraved = engrave_text(
#     clear,
#     "азъ буки веди\nглаголь добро есть",
#     font_size=48,
#     output_path=OUTPUT_DIR / "engraved_demo.png",
# )